# Claim Relationship Modeling Notebook

This notebook builds **two claim relationship graphs** and one **evolution trend chart** from your JSONL dataset:

1. **Semantic similarity graph** — claims that say similar things
2. **Performance relationship graph** — comparable numeric claims, where one outperforms another
3. **Dice-score evolution chart** — yearly development of Mean Dice and Best Dice

The evolution part is no longer a network graph; it is now a publication-year bar chart like the reference image.

The notebook is written to be **robust to missing values** in your dataset.


## Why these outputs?

Your dataset contains:
- `label` to identify claims
- `text` for claim content
- `year` for temporal analysis
- `structured_info.metric`, `structured_info.value`, and `structured_info.domain`

That means these are the most defensible outputs to establish:

### 1) Semantic similarity graph
Use this to group claims that are conceptually close even when wording differs.

### 2) Performance relationship graph
Use this only when claims are in the **same domain** and have comparable numeric evidence.

### 3) Dice-score evolution chart
Use publication year + extracted Dice values to show how reported Dice scores develop over time. This is easier for your thesis/professor to understand than an evolution network graph.


In [ ]:
# Change this path only if your dataset lives elsewhere
DATA_PATH = "/work/Research/MedClaimExtractor/outputs/t2/t2_structured_claims.jsonl"


In [ ]:
import json
import math
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Dataset exists:", Path(DATA_PATH).exists())

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

papers = load_jsonl(DATA_PATH)
print("Papers:", len(papers))

In [ ]:
# Quick structure check
sample = papers[0]
print("Top-level keys:", list(sample.keys()))
print("\nSample data:")
for key in sample.keys():
    if isinstance(sample[key], (list, dict)):
        print(f"  {key}: {type(sample[key]).__name__}")
        if isinstance(sample[key], list) and sample[key]:
            print(f"    - First item type: {type(sample[key][0]).__name__}")
            if isinstance(sample[key][0], dict):
                print(f"    - First item keys: {list(sample[key][0].keys())[:5]}")
    else:
        print(f"  {key}: {type(sample[key]).__name__}")

## Extract claims robustly

This version works with both common formats:

1. **Paper-level JSONL**: each paper has a `sentences` list, and each sentence has `label` and `structured_info`.
2. **Claim-level JSONL**: each line is already one claim sentence with `text` / `sentence` and `structured_info`.

It also keeps the required columns even if no claims are found, so `claims_df["metric"]` and `claims_df["domain"]` will not raise a `KeyError`.

In [ ]:
EXPECTED_CLAIM_COLUMNS = [
    "claim_id", "paper_idx", "sent_idx", "doi", "year", "title", "abstract",
    "text", "label", "model", "task", "dataset", "metric", "value",
    "comparison", "domain", "structured_info"
]

STRUCTURED_FIELDS = ["model", "task", "dataset", "metric", "value", "comparison", "domain"]

TRUE_LABELS = {1, "1", True, "true", "yes", "y", "claim", "claims", "positive", "check-worthy", "checkworthy"}
FALSE_LABELS = {0, "0", False, "false", "no", "n", "not claim", "non-claim", "nonclaim", "negative"}


def parse_json_if_needed(value):
    """Parse JSON strings when structured_info/value fields were saved as text."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        if (text.startswith("{") and text.endswith("}")) or (text.startswith("[") and text.endswith("]")):
            try:
                return json.loads(text)
            except Exception:
                return value
    return value


def first_non_empty(*values):
    for value in values:
        value = parse_json_if_needed(value)
        if value is None:
            continue
        if isinstance(value, float) and pd.isna(value):
            continue
        if isinstance(value, str) and value.strip() == "":
            continue
        return value
    return None


def get_any(d, keys, default=None):
    if not isinstance(d, dict):
        return default
    for key in keys:
        if key in d and d.get(key) not in [None, ""]:
            return d.get(key)
    return default


def normalize_structured_info(record):
    """Return a dictionary for structured_info, even when it is missing or stored as JSON text."""
    s_info = parse_json_if_needed(get_any(record, ["structured_info", "structure_info", "structured", "info"], {}))
    if isinstance(s_info, dict):
        return s_info
    return {}


def label_is_claim(label, assume_claim_when_missing=True):
    """Robust claim label check.

    If the label is missing, this function returns True because many T2 files are already
    claim-level JSONL files and do not repeat label=1 for every row.
    """
    if label is None or (isinstance(label, float) and pd.isna(label)):
        return assume_claim_when_missing

    if isinstance(label, str):
        norm = label.strip().lower()
    else:
        norm = label

    if norm in TRUE_LABELS:
        return True
    if norm in FALSE_LABELS:
        return False

    # Numeric fallback
    try:
        return int(label) == 1
    except Exception:
        return False


def make_claim_record(base_meta, sent, paper_idx, sent_idx):
    """Create one flat claim record from paper metadata + sentence/claim row."""
    if not isinstance(sent, dict):
        sent = {"text": str(sent)}

    s_info = normalize_structured_info(sent)

    # Metadata can appear either at paper level or claim/sentence level.
    doi = first_non_empty(get_any(sent, ["doi", "DOI"]), get_any(base_meta, ["doi", "DOI"]))
    year = first_non_empty(get_any(sent, ["year", "publication_year", "pub_year"]), get_any(base_meta, ["year", "publication_year", "pub_year"]))
    title = first_non_empty(get_any(sent, ["title", "paper_title"]), get_any(base_meta, ["title", "paper_title"]))
    abstract = first_non_empty(get_any(sent, ["abstract"]), get_any(base_meta, ["abstract"]))

    text = first_non_empty(
        get_any(sent, ["text", "sentence", "claim", "claim_text", "sent_text"]),
        get_any(s_info, ["text", "sentence", "claim", "claim_text"])
    )

    out = {
        "claim_id": None,
        "paper_idx": paper_idx,
        "sent_idx": sent_idx,
        "doi": doi,
        "year": year,
        "title": title,
        "abstract": abstract,
        "text": text,
        "label": get_any(sent, ["label", "is_claim", "claim_label"]),
        "structured_info": s_info,
    }

    for field in STRUCTURED_FIELDS:
        out[field] = first_non_empty(get_any(s_info, [field]), get_any(sent, [field]))

    return out


def extract_claims(rows):
    claims = []

    for paper_idx, row in enumerate(rows):
        if not isinstance(row, dict):
            continue

        # Common format 1: paper has a list of sentence dictionaries.
        if isinstance(row.get("sentences"), list):
            for sent_idx, sent in enumerate(row.get("sentences", [])):
                if not isinstance(sent, dict):
                    sent = {"text": str(sent)}
                label = get_any(sent, ["label", "is_claim", "claim_label"])
                if label_is_claim(label, assume_claim_when_missing=False):
                    claims.append(make_claim_record(row, sent, paper_idx, sent_idx))

        # Common format 2: paper has a list of already extracted claims.
        elif isinstance(row.get("claims"), list):
            for sent_idx, sent in enumerate(row.get("claims", [])):
                if not isinstance(sent, dict):
                    sent = {"text": str(sent)}
                label = get_any(sent, ["label", "is_claim", "claim_label"])
                if label_is_claim(label, assume_claim_when_missing=True):
                    claims.append(make_claim_record(row, sent, paper_idx, sent_idx))

        # Common format 3: each JSONL line is already a claim/sentence row.
        else:
            label = get_any(row, ["label", "is_claim", "claim_label"])
            if label_is_claim(label, assume_claim_when_missing=True):
                claims.append(make_claim_record({}, row, paper_idx, None))

    # IMPORTANT: keep columns even when claims list is empty.
    df = pd.DataFrame(claims, columns=EXPECTED_CLAIM_COLUMNS)
    if len(df) > 0:
        df["claim_id"] = range(len(df))
    else:
        df["claim_id"] = pd.Series(dtype="int")

    # Make sure all expected columns exist after any future edits.
    for col in EXPECTED_CLAIM_COLUMNS:
        if col not in df.columns:
            df[col] = None

    return df


claims_df = extract_claims(papers)
print("Extracted claims:", len(claims_df))
claims_df.head(3)


In [ ]:
print("Total claims:", len(claims_df))

print("\nMetric frequency:")
if "metric" in claims_df.columns and len(claims_df) > 0:
    print(claims_df["metric"].fillna("None").value_counts().head(15))
else:
    print("No metric values found. claims_df columns:", list(claims_df.columns))

print("\nDomain frequency:")
if "domain" in claims_df.columns and len(claims_df) > 0:
    print(claims_df["domain"].fillna("None").value_counts().head(15))
else:
    print("No domain values found. claims_df columns:", list(claims_df.columns))

if len(claims_df) == 0:
    print("\nWARNING: No claims were extracted.")
    print("This usually means your file uses a different label/field format, or the selected DATA_PATH is not the expected dataset.")
    print("Top-level sample keys:", list(papers[0].keys()) if len(papers) else [])
else:
    display(claims_df[["claim_id", "year", "text", "metric", "value", "domain"]].head(5))


## Numeric extraction for performance graph

Your dataset has many claims where `structured_info["value"]` is missing.
So we use a fallback strategy:

1. Use `structured_info["value"]` if it is numeric or a list of numerics
2. Otherwise extract percentages/decimals from the claim text
3. Keep a confidence flag so you know which scores were inferred

In [ ]:
number_pattern = re.compile(r'(?<!\d)(\d+(?:\.\d+)?)(?:\s*%)?')


def flatten_numeric_value(v):
    """Extract numeric values from numbers, lists, dictionaries, and strings."""
    v = parse_json_if_needed(v)
    if v is None:
        return []

    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return [float(v)]

    if isinstance(v, str):
        out = []
        for match in number_pattern.finditer(v):
            try:
                x = float(match.group(1))
                out.append(x)
            except Exception:
                pass
        return out

    if isinstance(v, (list, tuple, set)):
        out = []
        for x in v:
            out.extend(flatten_numeric_value(x))
        return out

    if isinstance(v, dict):
        out = []
        for x in v.values():
            out.extend(flatten_numeric_value(x))
        return out

    return []


def extract_numbers_from_text(text):
    nums = []
    text = str(text or "")
    for match in number_pattern.finditer(text):
        try:
            x = float(match.group(1))
            # convert likely percentages to 0-1 scale when appropriate
            if x > 1 and x <= 100 and "%" in text:
                x = x / 100.0
            nums.append(x)
        except Exception:
            pass
    return nums


higher_is_better_metrics = {
    "accuracy", "auc", "aoc", "dice", "precision", "recall",
    "sensitivity", "specificity", "f1", "f1-score", "iou",
    "balanced accuracy", "concordance index", "c-index", "overall survival"
}

lower_is_better_metrics = {
    "loss", "error", "mae", "mse", "rmse", "false positive rate", "hd95", "hausdorff"
}


def normalize_metric_name(metric):
    metric = parse_json_if_needed(metric)
    if metric is None or (isinstance(metric, float) and pd.isna(metric)):
        return None
    if isinstance(metric, (list, tuple, set)):
        metric = ", ".join(str(x) for x in metric if x is not None)
    if isinstance(metric, dict):
        metric = ", ".join(str(x) for x in metric.keys())

    m = str(metric).strip().lower()
    if not m:
        return None

    aliases = {
        "dice coefficient": "dice",
        "dice score": "dice",
        "dice similarity coefficient": "dice",
        "dsc": "dice",
        "acc": "accuracy",
        "auc score": "auc",
        "f1 score": "f1",
        "f1-score": "f1",
        "specificity, sensitivity": "diagnostic_pair",
        "sensitivity, specificity": "diagnostic_pair",
        "accuracy, auc": "combined_score",
        "precision, recall": "pr_pair",
    }
    return aliases.get(m, m)


def metric_direction(metric):
    m = normalize_metric_name(metric)
    if m in higher_is_better_metrics:
        return "higher"
    if m in lower_is_better_metrics:
        return "lower"
    if m in {"diagnostic_pair", "combined_score", "pr_pair"}:
        return "higher"
    return None


def performance_score(row):
    metric = normalize_metric_name(row.get("metric"))
    if metric_direction(metric) is None:
        return np.nan, None

    vals = flatten_numeric_value(row.get("value"))

    # Use explicit structured numeric values only for the performance graph.
    vals = [v for v in vals if 0 <= v <= 100]
    vals = [v / 100 if v > 1 else v for v in vals]

    if not vals:
        return np.nan, None

    if metric in {"diagnostic_pair", "combined_score", "pr_pair"} and len(vals) >= 2:
        score = float(np.mean(vals[:2]))
    else:
        score = float(np.mean(vals))

    return score, "structured_info"


if len(claims_df) == 0:
    claims_df["perf_score"] = pd.Series(dtype="float")
    claims_df["perf_source"] = pd.Series(dtype="object")
    claims_df["metric_norm"] = pd.Series(dtype="object")
else:
    perf_tmp = claims_df.apply(lambda r: pd.Series(performance_score(r)), axis=1)
    perf_tmp.columns = ["perf_score", "perf_source"]
    claims_df[["perf_score", "perf_source"]] = perf_tmp
    claims_df["metric_norm"] = claims_df["metric"].apply(normalize_metric_name)

claims_df[["text", "metric", "metric_norm", "value", "perf_score", "perf_source"]].head(10)


In [ ]:
if "perf_score" not in claims_df.columns:
    claims_df["perf_score"] = pd.Series(dtype="float")
if "perf_source" not in claims_df.columns:
    claims_df["perf_source"] = pd.Series(dtype="object")

print("Claims with usable performance score:", int(claims_df["perf_score"].notna().sum()))
print("By source:")
print(claims_df["perf_source"].fillna("none").value_counts())


## Relationship 1: Semantic similarity graph

We use TF-IDF + cosine similarity so this runs reliably anywhere.
If you want, you can later swap TF-IDF for sentence embeddings.

In [ ]:
SEMANTIC_THRESHOLD = 0.15
G_semantic = nx.Graph()

for _, row in claims_df.iterrows():
    G_semantic.add_node(
        int(row["claim_id"]),
        text=row.get("text"),
        year=row.get("year"),
        domain=row.get("domain"),
        metric=row.get("metric_norm"),
        title=row.get("title")
    )

if len(claims_df) < 2:
    print("Semantic graph: need at least 2 claims to compute similarities.")
else:
    texts = claims_df["text"].fillna("").astype(str)
    non_empty_texts = texts.str.strip().ne("").sum()

    if non_empty_texts < 2:
        print("Semantic graph: not enough non-empty claim text values.")
    else:
        try:
            min_df_value = 1 if len(claims_df) < 10 else 2
            tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=min_df_value, max_features=8000)
            X = tfidf.fit_transform(texts)
            semantic_sim = cosine_similarity(X)

            for i in range(len(claims_df)):
                for j in range(i + 1, len(claims_df)):
                    sim = float(semantic_sim[i, j])
                    if sim >= SEMANTIC_THRESHOLD:
                        G_semantic.add_edge(
                            int(claims_df.iloc[i]["claim_id"]),
                            int(claims_df.iloc[j]["claim_id"]),
                            weight=sim,
                            relation="semantic"
                        )
        except ValueError as e:
            print("Semantic graph could not be built:", e)

print("Semantic graph nodes:", G_semantic.number_of_nodes())
print("Semantic graph edges:", G_semantic.number_of_edges())


## Relationship 2: Performance graph

Rules:
- same domain when possible
- same normalized metric when possible
- both claims need a usable performance score
- create a directed edge from stronger claim → weaker claim

In [ ]:
def comparable_domain(d1, d2):
    if pd.isna(d1) or pd.isna(d2) or d1 is None or d2 is None:
        return False
    d1 = str(d1).strip().lower()
    d2 = str(d2).strip().lower()
    return bool(d1) and d1 == d2


def comparable_metric(m1, m2):
    if m1 is None or m2 is None:
        return False
    return normalize_metric_name(m1) == normalize_metric_name(m2)


PERFORMANCE_MARGIN = 0.00

G_performance = nx.DiGraph()
for _, row in claims_df.iterrows():
    G_performance.add_node(
        int(row["claim_id"]),
        text=row.get("text"),
        year=row.get("year"),
        domain=row.get("domain"),
        metric=row.get("metric_norm"),
        score=row.get("perf_score"),
        title=row.get("title")
    )

if len(claims_df) == 0:
    valid_perf = claims_df.copy()
else:
    valid_perf = claims_df[
        claims_df["perf_score"].notna() &
        claims_df["domain"].notna() &
        claims_df["metric_norm"].notna()
    ].copy()

for idx_i, row_i in valid_perf.iterrows():
    for idx_j, row_j in valid_perf.iterrows():
        if row_i["claim_id"] == row_j["claim_id"]:
            continue
        if not comparable_domain(row_i.get("domain"), row_j.get("domain")):
            continue
        if not comparable_metric(row_i.get("metric_norm"), row_j.get("metric_norm")):
            continue

        dir_i = metric_direction(row_i.get("metric_norm"))
        score_i = row_i.get("perf_score")
        score_j = row_j.get("perf_score")

        if pd.isna(score_i) or pd.isna(score_j):
            continue

        if dir_i == "higher" and score_i >= score_j + PERFORMANCE_MARGIN:
            G_performance.add_edge(
                int(row_i["claim_id"]),
                int(row_j["claim_id"]),
                weight=float(score_i - score_j),
                relation="outperforms"
            )
        elif dir_i == "lower" and score_i <= score_j - PERFORMANCE_MARGIN:
            G_performance.add_edge(
                int(row_i["claim_id"]),
                int(row_j["claim_id"]),
                weight=float(score_j - score_i),
                relation="outperforms"
            )

print("Performance graph nodes:", G_performance.number_of_nodes())
print("Performance graph edges:", G_performance.number_of_edges())
print("Performance graph connected nodes:", sum(1 for _, d in G_performance.degree() if d > 0))


## Relationship 3: Dice-score evolution chart

The evolution output is now a **bar chart**, not a network graph.

For each publication year, the notebook computes:
- **Mean Dice**: average Dice score for that year
- **Best Dice**: highest Dice score reported for that year
- **n_claims**: number of Dice-related claims used for that year

The chart is saved automatically under `outputs/t4/evolution_dice_by_year.png`.


In [ ]:
# Evolution analysis: Dice Score Development by Publication Year
# This replaces the old evolutionary network graph.

from pathlib import Path

DICE_TERMS = [
    "dice",
    "dsc",
    "dice coefficient",
    "dice similarity coefficient",
    "sørensen",
    "sorensen",
]


def serialize_for_search(value):
    """Convert metric/value objects into searchable text."""
    value = parse_json_if_needed(value)
    if value is None:
        return ""
    if isinstance(value, dict):
        return " ".join(f"{k} {serialize_for_search(v)}" for k, v in value.items())
    if isinstance(value, (list, tuple, set)):
        return " ".join(serialize_for_search(v) for v in value)
    return str(value)


def contains_dice_term(value):
    text = serialize_for_search(value).lower()
    return any(term in text for term in DICE_TERMS)


def extract_numeric_values(value):
    """Extract numeric values from int/float/list/dict/string values."""
    return flatten_numeric_value(value)


def normalize_dice_values(values):
    """Normalize Dice values to the 0-1 range and remove obvious non-Dice values."""
    normalized = []
    for x in values:
        if x is None or pd.isna(x):
            continue

        x = float(x)

        # Remove obvious years or impossible values.
        if 1900 <= x <= 2100:
            continue
        if x < 0 or x > 100:
            continue

        # Dice is often reported as 85.4 or 85.4%; convert to 0.854.
        if x > 1:
            x = x / 100.0

        if 0 <= x <= 1:
            normalized.append(x)

    return normalized


def extract_dice_score(row):
    """Return one representative Dice score for a claim, plus the extraction source."""
    metric_has_dice = contains_dice_term(row.get("metric")) or contains_dice_term(row.get("metric_norm"))
    text_has_dice = contains_dice_term(row.get("text"))

    # Use only claims that explicitly mention Dice in the metric field or sentence.
    if not (metric_has_dice or text_has_dice):
        return np.nan, None

    # Prefer structured_info.value.
    values = normalize_dice_values(extract_numeric_values(row.get("value")))
    source = "structured_info"

    # Fallback: extract from claim text only when the claim itself mentions Dice.
    if not values and text_has_dice:
        values = normalize_dice_values(extract_numeric_values(row.get("text")))
        source = "text_fallback"

    if not values:
        return np.nan, None

    # If a claim reports several Dice values, use their mean as the claim-level Dice.
    return float(np.mean(values)), source


if len(claims_df) == 0:
    claims_df["dice_score"] = pd.Series(dtype="float")
    claims_df["dice_source"] = pd.Series(dtype="object")
    claims_df["year_clean"] = pd.Series(dtype="float")
    dice_claims_df = claims_df.copy()
    evolution_summary_df = pd.DataFrame(columns=["year_clean", "mean_dice", "best_dice", "n_claims"])
else:
    dice_tmp = claims_df.apply(lambda r: pd.Series(extract_dice_score(r)), axis=1)
    dice_tmp.columns = ["dice_score", "dice_source"]
    claims_df[["dice_score", "dice_source"]] = dice_tmp

    claims_df["year_clean"] = pd.to_numeric(claims_df["year"], errors="coerce")

    dice_claims_df = claims_df[
        claims_df["dice_score"].notna() &
        claims_df["year_clean"].notna()
    ].copy()

    if len(dice_claims_df) > 0:
        dice_claims_df["year_clean"] = dice_claims_df["year_clean"].astype(int)
        evolution_summary_df = (
            dice_claims_df
            .groupby("year_clean", as_index=False)
            .agg(
                mean_dice=("dice_score", "mean"),
                best_dice=("dice_score", "max"),
                n_claims=("dice_score", "size"),
            )
            .sort_values("year_clean")
        )
    else:
        evolution_summary_df = pd.DataFrame(columns=["year_clean", "mean_dice", "best_dice", "n_claims"])

print("Dice-related claims used for evolution chart:", len(dice_claims_df))
display(evolution_summary_df)


def plot_dice_evolution_by_year(summary_df, save_path="outputs/t4/evolution_dice_by_year.png"):
    """Create a grouped bar chart like the reference image."""
    if summary_df.empty:
        print("No Dice values were found. Check metric/value annotations or lower the filtering rules.")
        return

    plot_df = summary_df.set_index("year_clean")[["mean_dice", "best_dice"]].rename(
        columns={"mean_dice": "Mean Dice", "best_dice": "Best Dice"}
    )

    ax = plot_df.plot(kind="bar", figsize=(14, 7), width=0.75)
    ax.set_title("Dice Score Development by Publication Year", fontsize=16)
    ax.set_xlabel("Publication year", fontsize=12)
    ax.set_ylabel("Dice score", fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.legend(loc="upper left")
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print("Saved evolution chart to:", save_path)
    plt.show()


plot_dice_evolution_by_year(evolution_summary_df)


## Enhanced graph drawing
This helper is used only for the **semantic** and **performance** relationship graphs.

The evolution output is handled separately as a Dice-score bar chart.


In [ ]:
def draw_relationship_graph_enhanced(
    G,
    title,
    max_nodes=120,
    figsize=(14, 10),
    seed=42,
    connected_only=True,
):
    if G is None:
        raise ValueError(f"{title}: graph is None")

    if G.number_of_nodes() == 0:
        print(f"{title}: graph has no nodes")
        return

    if G.number_of_edges() == 0:
        print(f"{title}: graph has nodes but no edges. Try lowering the threshold.")
        return

    H = G.copy()

    # Show only nodes that have at least one edge. This removes isolated nodes.
    if connected_only:
        connected_nodes = [n for n, d in H.degree() if d > 0]
        H = H.subgraph(connected_nodes).copy()

    if H.number_of_nodes() == 0 or H.number_of_edges() == 0:
        print(f"{title}: no connected nodes to visualize")
        return

    # If there are too many connected nodes, keep the most connected ones.
    if H.number_of_nodes() > max_nodes:
        degrees = sorted(H.degree, key=lambda x: x[1], reverse=True)
        keep_nodes = [n for n, _ in degrees[:max_nodes]]
        H = H.subgraph(keep_nodes).copy()

    if isinstance(H, nx.DiGraph):
        n_components = nx.number_weakly_connected_components(H)
    else:
        n_components = nx.number_connected_components(H)

    plt.figure(figsize=figsize)
    pos = nx.spring_layout(H, seed=seed, k=0.55, iterations=200)

    if isinstance(H, nx.DiGraph):
        nx.draw_networkx_edges(H, pos, arrows=True, alpha=0.35, width=1.0, arrowsize=12)
    else:
        nx.draw_networkx_edges(H, pos, alpha=0.35, width=1.0)

    nx.draw_networkx_nodes(H, pos, node_size=140, alpha=0.90)
    labels = {n: f"{n}\n{H.nodes[n].get('year', '')}" for n in H.nodes()}
    nx.draw_networkx_labels(H, pos, labels=labels, font_size=7)

    plt.title(f"{title}\nvisualized nodes={H.number_of_nodes()}, edges={H.number_of_edges()}, groups={n_components}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
draw_relationship_graph_enhanced(
    G_semantic,
    "Semantic relationship graph",
    connected_only=True,
    max_nodes=80,
    figsize=(14, 10)
)

draw_relationship_graph_enhanced(
    G_performance,
    "Performance relationship graph",
    connected_only=True,
    max_nodes=200,
    figsize=(14, 10)
)


## Top relationships and evolution table to inspect

These tables help you interpret the strongest semantic/performance edges and the yearly Dice-score evolution.


In [ ]:
def graph_edges_to_df(G, top_n=20):
    rows = []
    if G is None or G.number_of_edges() == 0:
        return pd.DataFrame()

    for u, v, data in G.edges(data=True):
        rows.append({
            "source_id": u,
            "target_id": v,
            "weight": data.get("weight"),
            "relation": data.get("relation"),
            "source_year": G.nodes[u].get("year"),
            "target_year": G.nodes[v].get("year"),
            "source_domain": G.nodes[u].get("domain"),
            "target_domain": G.nodes[v].get("domain"),
            "source_metric": G.nodes[u].get("metric"),
            "target_metric": G.nodes[v].get("metric"),
            "source_text": G.nodes[u].get("text"),
            "target_text": G.nodes[v].get("text"),
        })
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return out
    return out.sort_values("weight", ascending=False).head(top_n)


semantic_edges_df = graph_edges_to_df(G_semantic, top_n=20)
performance_edges_df = graph_edges_to_df(G_performance, top_n=20)

print("Top semantic edges")
if semantic_edges_df.empty:
    print("No semantic edges found. Try lowering SEMANTIC_THRESHOLD or check claim text extraction.")
else:
    display(semantic_edges_df[["source_id", "target_id", "weight", "source_text", "target_text"]])

print("Top performance edges")
if performance_edges_df.empty:
    print("No performance edges found. Check domain/metric/value coverage or lower PERFORMANCE_MARGIN.")
else:
    display(performance_edges_df[["source_id", "target_id", "weight", "source_domain", "source_metric", "source_text", "target_text"]])

print("Dice evolution summary")
if evolution_summary_df.empty:
    print("No Dice evolution summary available.")
else:
    display(evolution_summary_df)


## Optional: Combined relationship edge table

This creates a unified edge table for the two graph-based relationship types:
- semantic
- performance

The Dice evolution output is a separate yearly summary table and bar chart, so it is not included as an edge table.


In [ ]:
def make_edge_table():
    frames = []
    for name, G in [
        ("semantic", G_semantic),
        ("performance", G_performance),
    ]:
        rows = []
        if G is not None:
            for u, v, data in G.edges(data=True):
                rows.append({
                    "relation_type": name,
                    "source": u,
                    "target": v,
                    "weight": data.get("weight"),
                    "source_year": G.nodes[u].get("year"),
                    "target_year": G.nodes[v].get("year"),
                    "source_domain": G.nodes[u].get("domain"),
                    "target_domain": G.nodes[v].get("domain"),
                    "source_text": G.nodes[u].get("text"),
                    "target_text": G.nodes[v].get("text"),
                })
        frames.append(pd.DataFrame(rows))

    frames = [f for f in frames if not f.empty]
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


edge_table = make_edge_table()
edge_table.head()


## Interpretation guide for your thesis or paper

### Semantic relationship graph
Write:
> Claims are semantically related when their textual content expresses similar findings, methods, or conclusions. This relationship was operationalized using TF-IDF representations and cosine similarity.

### Performance relationship graph
Write:
> Claims are performance-related when they concern the same domain and metric, and one claim reports stronger quantitative evidence than another. This relationship was operationalized as a directed edge from the higher-performing claim to the lower-performing claim.

### Dice-score evolution chart
Write:
> The yearly Dice-score chart summarizes how segmentation performance develops across publication years. Mean Dice shows the average reported Dice value for each year, while Best Dice shows the strongest reported Dice value in that year.

### Important limitation
Write:
> The Dice evolution chart summarizes reported values extracted from claims. It should be interpreted as a literature-level trend, not as a controlled benchmark, because papers may use different datasets, tumor regions, preprocessing pipelines, and validation settings.


## Strong recommendation for your final analysis chapter

Use these three outputs:
- **semantic similarity graph** for clustering related literature claims
- **performance relationship graph** for comparing evidence strength
- **Dice-score evolution bar chart** for showing development over publication years

This combination is stronger and clearer than using an evolution network graph, because the yearly Dice chart is easier to interpret in thesis figures and presentations.
